In [1]:
from datetime import datetime, timedelta
import faiss
from langchain_classic.docstore import InMemoryDocstore
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH10-Retriever")

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model, index, InMemoryDocstore({}),{})
#시간 가중치가 적용된 벡터 스토어 리트리버를 초기화 ( 여기서는 낮은 감쇠율을 적용 )
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,decay_rate=0.0000000000000000000000001, k=1
)

LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever


In [ ]:
yesterday = datetime.now() - timedelta(days=1)

retriever.add_documents(
    [
        Document(
            page_content="Isaac을 불러주세요.",
            metadata={"last_accessed_at": yesterday},
        )
    ]
)


['aacc5d39-ae3c-4bb7-a832-aaf6ddd21ec3']

In [5]:
retriever.add_documents([Document(page_content="테디 노트는 누구야")])

['55ee170f-2a87-4828-8f23-4a3fabc9d2ba']

In [6]:
retriever.invoke("아이작")

d:\rag_one\.venv\Lib\site-packages\langchain_classic\retrievers\time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='aacc5d39-ae3c-4bb7-a832-aaf6ddd21ec3', metadata={'last_accessed_at': datetime.datetime(2026, 9, 22, 17, 6, 39, 820250), 'created_at': datetime.datetime(2026, 9, 23, 17, 6, 39, 820250), 'buffer_idx': 0}, page_content='Isaac을 불러주세요.'), np.float32(-0.051826715)), (Document(id='55ee170f-2a87-4828-8f23-4a3fabc9d2ba', metadata={'last_accessed_at': datetime.datetime(2026, 9, 23, 17, 8, 7, 13104), 'created_at': datetime.datetime(2026, 9, 23, 17, 8, 7, 13104), 'buffer_idx': 1}, page_content='테디 노트는 누구야'), np.float32(-0.16925824))]
  docs_and_scores = self.vectorstore.similarity_search_with_relevance_scores(


[Document(metadata={'last_accessed_at': datetime.datetime(2026, 9, 23, 17, 8, 12, 273170), 'created_at': datetime.datetime(2026, 9, 23, 17, 6, 39, 820250), 'buffer_idx': 0}, page_content='Isaac을 불러주세요.')]

In [7]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

#벡터스토어를 빈 상태로 초기화
embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore=FAISS(embeddings_model, index, InMemoryDocstore({}),{})

#시간 가중치가 적용된 벡터 스토어 리트리버를 초기화
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore, decay_rate=0.999, k=1  # 감쇠율을 1로 설정하면 모든객체의 recency시 값이 0이되어 vector lookup과 동일한 결과를 얻게됩니다.
)

In [8]:
yesterday = datetime.now() - timedelta(days=1)

retriever.add_documents(
    [
        Document(
            page_content="Isaac을 찾아주세요.",
            metadata={"last_accessed_at": yesterday},
        )
    ]
)

['bf1b9cde-9a73-4a8f-9f4e-0f3ec7a3ea96']

In [12]:
retriever.add_documents([Document(page_content="Isaac은 윗층에 있어요.")])

['3483f90c-5734-4be4-9bb3-c144e030544f']

In [13]:
retriever.invoke("isacc")

d:\rag_one\.venv\Lib\site-packages\langchain_classic\retrievers\time_weighted_retriever.py:86: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='bf1b9cde-9a73-4a8f-9f4e-0f3ec7a3ea96', metadata={'last_accessed_at': datetime.datetime(2026, 9, 22, 17, 12, 51, 295491), 'created_at': datetime.datetime(2026, 9, 23, 17, 12, 51, 295491), 'buffer_idx': 0}, page_content='Isaac을 찾아주세요.'), np.float32(0.08184892)), (Document(id='3483f90c-5734-4be4-9bb3-c144e030544f', metadata={'last_accessed_at': datetime.datetime(2026, 9, 23, 17, 14, 25, 473770), 'created_at': datetime.datetime(2026, 9, 23, 17, 14, 25, 473770), 'buffer_idx': 2}, page_content='Isaac은 윗층에 있어요.'), np.float32(-0.025178194)), (Document(id='1ffe561f-7b2a-49f0-af43-0ce8518263b5', metadata={'last_accessed_at': datetime.datetime(2026, 9, 23, 17, 13, 32, 363912), 'created_at': datetime.datetime(2026, 9, 23, 17, 13, 32, 363912), 'buffer_idx': 1}, page_content='테디는 누구고 테디노트는 누구야?'), np.float32(-0.32931566))]
  docs_and

[Document(metadata={'last_accessed_at': datetime.datetime(2026, 9, 23, 17, 14, 27, 791552), 'created_at': datetime.datetime(2026, 9, 23, 17, 14, 25, 473770), 'buffer_idx': 2}, page_content='Isaac은 윗층에 있어요.')]

In [15]:
import datetime
from langchain_classic.utils import mock_now

mock_now(datetime.datetime(2024,8,30,00,00))

print(datetime.datetime.now())

2026-09-23 17:15:55.816170


In [ ]:
with mock_now(datetime.datetime(2026,9,23,00,00)):
    print(retriever.invoke("isaac"))

OverflowError: (34, 'Result too large')